# MCP + LangChain Agent — Smart Travel Planner (Agentic)

> Pinned to `venv-agents-requirements-lock.txt` — MCP **1.26.0**, LangChain **1.3.9**, LangGraph **1.2.5**.

Companion to `mcp_langgraph_travel_planner.ipynb`. Same MCP tools and data sources — **different orchestration**:

| | Pipeline notebook | This notebook (agentic) |
|---|---|---|
| Orchestration | Fixed 5-node LangGraph pipeline | **LangChain `create_agent`** (ReAct loop) |
| Tool selection | Hard-coded per node | **LLM decides** which tools to call |
| Tool order | Always research → weather → budget | Dynamic (agent may skip or reorder) |
| MCP role | Called explicitly in nodes 2–4 | Called via **LangChain `@tool` wrappers** |

## Architecture

```
User Query: "Plan a 5-day trip to Tokyo"
        │
        ▼
create_agent (LangChain 1.x)  <── InMemorySaver (optional thread memory)
        │
        ├── think → call research_destination (MCP :8011)
        ├── think → call get_weather          (MCP :8011)
        ├── think → call get_travel_costs     (MCP :8011)
        └── synthesize final answer
        │
        ▼
📋 Travel plan (agent-written)
```

## Prerequisites
- Python 3.10+
- `OPENAI_API_KEY` in `.env`
- No OpenWeather key needed (uses wttr.in like the pipeline notebook)

## Step 1: Install Dependencies

Pinned versions from `venv-agents-requirements-lock.txt`. Run once in a fresh environment.

In [ ]:
# Run once in a fresh environment
#%pip install \
#    mcp==1.26.0 \
#    nest-asyncio==1.6.0 \
#    uvicorn==0.38.0 \
#    starlette==0.47.0 \
#    sse-starlette==2.3.6 \
#    httpx==0.28.1 \
#    pydantic==2.12.5 \
#    langchain==1.3.9 \
#    langchain-openai==1.3.2 \
#    langchain-core==1.4.7 \
#    langgraph==1.2.5 \
#    langgraph-prebuilt==1.1.0 \
#    langsmith==0.8.16 \
#    openai==2.42.0 \
#    python-dotenv==1.2.2 \
#    wikipedia==1.4.0

## Step 2: Imports & Configuration

In [1]:
# asyncio is required to run two or more events in parallel. 
import nest_asyncio
nest_asyncio.apply()

import asyncio
import logging
import os
import socket
import threading
import time
import urllib.parse
from typing import Any, Dict, List, Optional

# httpx to make client - server connection
import httpx

# uvcorn is nothing but an ASGI - asynchronous Gateway Interface - used nowadays to host MCP server
# keeps server running and ready to accept network requests
# client may connect to MCP servers on SSE (server sent events), http, Web Sockets
import uvicorn

import wikipedia
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.sse import sse_client

# FastMCP class will be used to create MCP server  
from mcp.server.fastmcp import FastMCP

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

for _lg in ["uvicorn", "uvicorn.access", "httpx", "openai", "httpcore", "mcp"]:
    logging.getLogger(_lg).setLevel(logging.ERROR)

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

MODEL_NAME = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL_NAME, temperature=0.3)
print(f"Configuration loaded — model: {MODEL_NAME}")

Configuration loaded — model: gpt-4o-mini


## Step 3: Travel Cost Database

Same offline `COST_DB` as the pipeline notebook — used by the `get_travel_costs` MCP tool.

In [2]:
COST_DB: Dict[str, Dict] = {
    "paris": {"hotel": 150, "food": 60, "transport": 20, "activities": 40, "currency": "EUR", "fx": 0.92},
    "tokyo": {"hotel": 120, "food": 50, "transport": 25, "activities": 35, "currency": "JPY", "fx": 150},
    "new york": {"hotel": 220, "food": 80, "transport": 15, "activities": 55, "currency": "USD", "fx": 1.0},
    "london": {"hotel": 180, "food": 65, "transport": 20, "activities": 45, "currency": "GBP", "fx": 0.79},
    "bali": {"hotel": 60, "food": 25, "transport": 12, "activities": 25, "currency": "IDR", "fx": 15600},
    "dubai": {"hotel": 160, "food": 70, "transport": 18, "activities": 50, "currency": "AED", "fx": 3.67},
    "singapore": {"hotel": 140, "food": 55, "transport": 20, "activities": 40, "currency": "SGD", "fx": 1.34},
    "rome": {"hotel": 130, "food": 55, "transport": 15, "activities": 35, "currency": "EUR", "fx": 0.92},
    "barcelona": {"hotel": 120, "food": 50, "transport": 15, "activities": 35, "currency": "EUR", "fx": 0.92},
    "amsterdam": {"hotel": 145, "food": 60, "transport": 18, "activities": 40, "currency": "EUR", "fx": 0.92},
    "default": {"hotel": 100, "food": 45, "transport": 18, "activities": 30, "currency": "USD", "fx": 1.0},
}
print(f"COST_DB loaded — {len(COST_DB) - 1} cities + default")

COST_DB loaded — 10 cities + default


## Step 4: MCP Server (FastMCP)

Identical tool implementations to the pipeline notebook. Uses port **8011** so it can run alongside the pipeline server on 8010.

In [3]:
travel_mcp = FastMCP(
    name="Travel Tools MCP Server",
    instructions="Travel toolkit: research_destination, get_weather, get_travel_costs.",
)


@travel_mcp.tool()
async def research_destination(destination: str) -> str:
    """Research a travel destination via Wikipedia. Returns overview text."""
    query = (destination or "").strip()
    if not query:
        return "DESTINATION: Unknown\n\nError: empty destination"

    # Robust fetch path: retry Wikipedia API first, then fallback to wikipedia package
    for attempt in range(2):
        try:
            async with httpx.AsyncClient(timeout=15.0, verify=False) as client:
                # 1) Search best page title from Wikipedia API
                search_resp = await client.get(
                    "https://en.wikipedia.org/w/api.php",
                    params={
                        "action": "query",
                        "list": "search",
                        "srsearch": query,
                        "utf8": 1,
                        "format": "json",
                    },
                    headers={"User-Agent": "MCP-Travel-Agent/1.0"},
                )
                search_resp.raise_for_status()
                search_json = search_resp.json()
                hits = search_json.get("query", {}).get("search", [])
                title = hits[0]["title"] if hits else query

                # 2) Fetch page summary for the chosen title
                safe_title = urllib.parse.quote(title, safe="")
                summary_resp = await client.get(
                    f"https://en.wikipedia.org/api/rest_v1/page/summary/{safe_title}",
                    headers={"User-Agent": "MCP-Travel-Agent/1.0"},
                )
                summary_resp.raise_for_status()
                page_json = summary_resp.json()

                extract = (page_json.get("extract") or "").strip()
                final_title = page_json.get("title", title)

                if extract:
                    return f"WIKIPEDIA OVERVIEW — {final_title.upper()}\n\n{extract}"

        except Exception:
            if attempt == 0:
                await asyncio.sleep(0.35)
                continue

    # Fallback path: wikipedia package
    loop = asyncio.get_event_loop()
    try:
        summary = await loop.run_in_executor(
            None,
            lambda: wikipedia.summary(query, sentences=8, auto_suggest=True),
        )
        return f"WIKIPEDIA OVERVIEW — {query.upper()}\n\n{summary}"
    except wikipedia.exceptions.DisambiguationError as e:
        try:
            summary = await loop.run_in_executor(
                None, lambda: wikipedia.summary(e.options[0], sentences=8)
            )
            return f"WIKIPEDIA OVERVIEW — {e.options[0].upper()}\n\n{summary}"
        except Exception:
            return f"DESTINATION: {destination}\n\nPopular travel destination."
    except Exception as exc:
        return f"DESTINATION: {destination}\n\nError: {str(exc)[:160]}"


@travel_mcp.tool()
async def get_weather(city: str) -> str:
    """Fetch current weather for a city from wttr.in (no API key)."""
    try:
        async with httpx.AsyncClient(timeout=12.0, verify=False) as client:
            r = await client.get(
                f"https://wttr.in/{city}?format=j1",
                headers={"User-Agent": "MCP-Travel-Agent/1.0"},
            )
        if r.status_code != 200:
            return f"Weather unavailable (HTTP {r.status_code}) for '{city}'."
        data = r.json()
        cur = data["current_condition"][0]
        country = data["nearest_area"][0]["country"][0]["value"]
        return (
            f"CURRENT WEATHER — {city.upper()}, {country}\n"
            f"Temperature: {cur['temp_C']}°C | Humidity: {cur['humidity']}%\n"
            f"Conditions: {cur['weatherDesc'][0]['value']}"
        )
    except Exception as exc:
        return f"WEATHER UNAVAILABLE for '{city}': {str(exc)[:120]}"


@travel_mcp.tool()
def get_travel_costs(destination: str, days: int = 5) -> str:
    """Return mid-range daily travel cost estimates for a destination and trip length."""
    key = destination.lower().strip()
    costs = COST_DB.get(key, COST_DB["default"])
    daily = costs["hotel"] + costs["food"] + costs["transport"] + costs["activities"]
    total = daily * days
    note = "exact city data" if key in COST_DB else "global average"
    return (
        f"COST ESTIMATE — {destination.upper()} ({days} days) [{note}]\n"
        f"Daily: ${daily}/day | Trip total: ${total} USD"
    )


print("FastMCP tools defined: research_destination, get_weather, get_travel_costs")

FastMCP tools defined: research_destination, get_weather, get_travel_costs


c:\Divya\code\CapStone\.venv\Lib\site-packages\pydantic_settings\sources\utils.py:47: IncompleteFieldDefinitionWarning: Field 'lifespan' has an incomplete definition: its annotation contains an unresolved forward reference, so settings sources may fail to correctly resolve its value. Call `model_rebuild()` on the model where the field is defined, once all the referenced types are defined.
  warnings.warn(


## Step 5: Start MCP Server (background thread)

In [4]:
MCP_HOST = "127.0.0.1"
MCP_PORT = 8011


def _port_in_use(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.2)
        return s.connect_ex((host, port)) == 0


# If previous runs already occupy 8011, move to the next free port so new code is used.
if _port_in_use(MCP_HOST, MCP_PORT):
    for p in range(8012, 8021):
        if not _port_in_use(MCP_HOST, p):
            MCP_PORT = p
            break

MCP_BASE_URL = f"http://{MCP_HOST}:{MCP_PORT}"

try:
    mcp_asgi_app = travel_mcp.sse_app()
except AttributeError:
    mcp_asgi_app = travel_mcp.streamable_http_app()


def _run_mcp_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        config = uvicorn.Config(mcp_asgi_app, host=MCP_HOST, port=MCP_PORT, log_level="error")
        server = uvicorn.Server(config)
        loop.run_until_complete(server.serve())
    finally:
        loop.close()


threading.Thread(target=_run_mcp_server, daemon=True).start()
print(f"Starting MCP server on {MCP_BASE_URL} ...")
time.sleep(3)
print("MCP server ready.")

Starting MCP server on http://127.0.0.1:8011 ...
MCP server ready.


## Step 6: MCP Client + LangChain Tool Wrappers

The agent does **not** call MCP directly. LangChain `@tool` functions delegate to `MCPClient` —
bridging the agentic loop with the MCP tool server.

In [5]:
class MCPClient:
    """Minimal async MCP client over SSE transport."""

    def __init__(self, base_url: str):
        self.sse_url = f"{base_url}/sse"

    async def call_tool(self, tool_name: str, arguments: Dict[str, Any]) -> str:
        async with sse_client(self.sse_url) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(tool_name, arguments)
                return "\n".join(
                    item.text for item in result.content if hasattr(item, "text")
                )


mcp_client = MCPClient(MCP_BASE_URL)

### Below we are creating langchain compatible tools - three of them - and calling them like mcp_client.call_tool(....)
### mcp_client is an instance defined just above from MCPClient Class. This instance is passed with MCP server base URL.

@tool
async def research_destination(destination: str) -> str:
    """Research a travel destination. Use for attractions, culture, history, and visitor tips."""
    return await mcp_client.call_tool("research_destination", {"destination": destination})


@tool
async def get_weather(city: str) -> str:
    """Get current weather for a city. Use for packing advice and outdoor timing."""
    return await mcp_client.call_tool("get_weather", {"city": city})


@tool
async def get_travel_costs(destination: str, days: int = 5) -> str:
    """Estimate mid-range daily travel costs for a destination and number of days."""
    return await mcp_client.call_tool(
        "get_travel_costs", {"destination": destination, "days": days}
    )


tools = [research_destination, get_weather, get_travel_costs]
print(f"LangChain tools ready ({len(tools)}) — each delegates to MCP server on :{MCP_PORT}")

LangChain tools ready (3) — each delegates to MCP server on :8011


## Step 7: Build the Travel Agent (`create_agent`)

LangChain 1.x `create_agent` runs a **ReAct loop**: the LLM reasons, picks tools, reads results, and repeats until it can answer.

In [6]:
SYSTEM_PROMPT = """You are TravelBot, an expert AI travel planner.

When a user asks for trip planning help:
1. Identify the destination and trip duration from their message.
2. Call research_destination to learn about the place.
3. Call get_weather for current conditions and packing advice.
4. Call get_travel_costs with the destination and number of days.
5. Synthesize a comprehensive travel plan with sections:
   - Trip Highlights
   - Suggested Itinerary
   - Weather & Packing
   - Budget Breakdown
   - Essential Tips

Always use the MCP tools before writing your final answer.
Be practical, specific, and concise."""

checkpointer = InMemorySaver()
travel_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

SESSION_CONFIG = {"configurable": {"thread_id": "travel-agent-001"}}
print("Travel agent ready (create_agent + MCP tools + InMemorySaver).")

Travel agent ready (create_agent + MCP tools + InMemorySaver).


## Step 8: Helper — Run Agent & Show Tool Trace

In [7]:
def print_tool_trace(messages: List) -> None:
    """Print which tools the agent called and matched results."""
    print("\n--- Agent Tool Trace ---")
    step = 0
    pending_calls: Dict[str, str] = {}

    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                step += 1
                call_id = tc.get("id")
                args = tc.get("args", tc.get("arguments", {}))
                pending_calls[call_id] = tc["name"]
                print(f"  {step}. {tc['name']}({args})")

        elif isinstance(msg, ToolMessage):
            call_id = getattr(msg, "tool_call_id", None)
            tool_name = pending_calls.get(call_id, "unknown_tool")
            content = msg.content
            if isinstance(content, list):
                content = " ".join(
                    part.get("text", str(part)) if isinstance(part, dict) else str(part)
                    for part in content
                )
            preview = (str(content) if content is not None else "")[:120].replace("\n", " ")
            print(f"     → [{tool_name}] result: {preview}...")

    if step == 0:
        print("  (no tool calls recorded)")
    print("---\n")


async def plan_trip(query: str, thread_id: str = "travel-agent-001") -> str:
    """Send a query to the travel agent; return the final plan."""
    config = {"configurable": {"thread_id": thread_id}}
    print(f"\n{'=' * 60}")
    print(f"User: {query}")
    print(f"{'=' * 60}")

    t0 = time.perf_counter()
    result = await travel_agent.ainvoke(
        {"messages": [HumanMessage(content=query)]},
        config=config,
    )
    elapsed = time.perf_counter() - t0

    print_tool_trace(result["messages"])
    answer = result["messages"][-1].content
    print(f"TravelBot ({elapsed:.1f}s):\n{answer}\n")
    return answer

## Step 9: Demo — Agentic Travel Planning

In [8]:
#await plan_trip("Plan a 8-day trip to Chandigadh")
await plan_trip("Plan a 8-day trip to Dubai")


User: Plan a 8-day trip to Dubai

--- Agent Tool Trace ---
  1. research_destination({'destination': 'Dubai'})
  2. get_weather({'city': 'Dubai'})
  3. get_travel_costs({'destination': 'Dubai', 'days': 8})
     → [research_destination] result: WIKIPEDIA OVERVIEW — DUBAI  Dubai is the most populous city in the United Arab Emirates and the capital of the Emirate o...
     → [get_weather] result: CURRENT WEATHER — DUBAI, United Arab Emirates Temperature: 34°C | Humidity: 49% Conditions: Sunny...
     → [get_travel_costs] result: COST ESTIMATE — DUBAI (8 days) [exact city data] Daily: $298/day | Trip total: $2384 USD...
---

TravelBot (7.2s):
Here's a comprehensive travel plan for your 8-day trip to Dubai:

### Trip Highlights
- **Burj Khalifa**: Visit the tallest building in the world for breathtaking views.
- **Dubai Mall**: Explore one of the largest shopping malls, featuring an aquarium and ice rink.
- **Desert Safari**: Experience dune bashing, camel rides, and traditional Bedouin cu

"Here's a comprehensive travel plan for your 8-day trip to Dubai:\n\n### Trip Highlights\n- **Burj Khalifa**: Visit the tallest building in the world for breathtaking views.\n- **Dubai Mall**: Explore one of the largest shopping malls, featuring an aquarium and ice rink.\n- **Desert Safari**: Experience dune bashing, camel rides, and traditional Bedouin culture.\n- **Palm Jumeirah**: Relax on this iconic man-made island and visit the luxurious Atlantis resort.\n- **Dubai Marina**: Enjoy the vibrant waterfront with dining and entertainment options.\n- **Old Dubai**: Discover the historic district, including the Dubai Museum and spice souks.\n\n### Suggested Itinerary\n**Day 1**: Arrival in Dubai, check into your hotel, and relax.  \n**Day 2**: Visit Burj Khalifa and Dubai Mall. Evening fountain show.  \n**Day 3**: Desert Safari with dinner under the stars.  \n**Day 4**: Explore Palm Jumeirah and relax at the beach.  \n**Day 5**: Visit Dubai Marina and take a boat tour.  \n**Day 6**: Dis

In [9]:
await plan_trip("I want to visit Rome for 4 days — what should I budget and pack?", thread_id="travel-agent-rome")


User: I want to visit Rome for 4 days — what should I budget and pack?

--- Agent Tool Trace ---
  1. research_destination({'destination': 'Rome'})
  2. get_weather({'city': 'Rome'})
  3. get_travel_costs({'destination': 'Rome', 'days': 4})
     → [research_destination] result: WIKIPEDIA OVERVIEW — ROME  Rome is the capital city and most populated comune (municipality) of Italy. It is also the ad...
     → [get_weather] result: CURRENT WEATHER — ROME, Togo Temperature: 24°C | Humidity: 92% Conditions: Cloudy ...
     → [get_travel_costs] result: COST ESTIMATE — ROME (4 days) [exact city data] Daily: $235/day | Trip total: $940 USD...
---

TravelBot (7.3s):
Here's a comprehensive travel plan for your 4-day trip to Rome:

### Trip Highlights
- **Historical Landmarks**: Explore the Colosseum, Roman Forum, and Palatine Hill.
- **Vatican City**: Visit St. Peter's Basilica, the Vatican Museums, and the Sistine Chapel.
- **Cultural Experiences**: Enjoy authentic Italian cuisine, gelato, and 

"Here's a comprehensive travel plan for your 4-day trip to Rome:\n\n### Trip Highlights\n- **Historical Landmarks**: Explore the Colosseum, Roman Forum, and Palatine Hill.\n- **Vatican City**: Visit St. Peter's Basilica, the Vatican Museums, and the Sistine Chapel.\n- **Cultural Experiences**: Enjoy authentic Italian cuisine, gelato, and local markets.\n- **Parks and Piazzas**: Relax in Villa Borghese and visit iconic squares like Piazza Navona and the Spanish Steps.\n\n### Suggested Itinerary\n**Day 1**: Arrival and explore the Colosseum and Roman Forum. Dinner in Trastevere.\n**Day 2**: Visit Vatican City - St. Peter's Basilica and Vatican Museums. Evening stroll in the Vatican Gardens.\n**Day 3**: Day trip to the ancient city of Ostia Antica or explore the Borghese Gallery. Enjoy a cooking class in the evening.\n**Day 4**: Leisurely morning at Villa Borghese, followed by shopping in the city center. Departure.\n\n### Weather & Packing\n- **Current Weather**: 24°C, Cloudy, Humidity: 

In [10]:
await plan_trip("I want to visit Paris for 4 days — what should I budget and pack?", thread_id="travel-agent-paris")


User: I want to visit Paris for 4 days — what should I budget and pack?

--- Agent Tool Trace ---
  1. research_destination({'destination': 'Paris'})
  2. get_weather({'city': 'Paris'})
  3. get_travel_costs({'destination': 'Paris', 'days': 4})
     → [research_destination] result: WIKIPEDIA OVERVIEW — PARIS  Paris is the capital and largest city of France, with an estimated city population of 2.04 m...
     → [get_weather] result: CURRENT WEATHER — PARIS, France Temperature: 20°C | Humidity: 69% Conditions: Sunny...
     → [get_travel_costs] result: COST ESTIMATE — PARIS (4 days) [exact city data] Daily: $270/day | Trip total: $1080 USD...
---

TravelBot (5.8s):
Here's a comprehensive travel plan for your 4-day trip to Paris:

### Trip Highlights
- **Eiffel Tower**: Iconic symbol of Paris, offering stunning views.
- **Louvre Museum**: Home to thousands of works of art, including the Mona Lisa.
- **Notre-Dame Cathedral**: A masterpiece of French Gothic architecture.
- **Montmartre**: 

"Here's a comprehensive travel plan for your 4-day trip to Paris:\n\n### Trip Highlights\n- **Eiffel Tower**: Iconic symbol of Paris, offering stunning views.\n- **Louvre Museum**: Home to thousands of works of art, including the Mona Lisa.\n- **Notre-Dame Cathedral**: A masterpiece of French Gothic architecture.\n- **Montmartre**: A charming neighborhood known for its artistic history and the Basilica of Sacré-Cœur.\n- **Seine River Cruise**: A relaxing way to see the city from the water.\n\n### Suggested Itinerary\n**Day 1**: \n- Arrive in Paris, check into your accommodation.\n- Visit the Eiffel Tower and enjoy the views.\n- Dinner at a nearby café.\n\n**Day 2**: \n- Morning at the Louvre Museum.\n- Afternoon stroll through the Tuileries Garden.\n- Evening in Montmartre, visit the Basilica of Sacré-Cœur.\n\n**Day 3**: \n- Explore Notre-Dame Cathedral and Île de la Cité.\n- Afternoon shopping in the Marais district.\n- Dinner in a traditional French bistro.\n\n**Day 4**: \n- Morning 

In [11]:
# Agent can handle follow-up in the same thread (InMemorySaver memory)
await plan_trip("Make that itinerary more budget-friendly", thread_id="travel-agent-001")


User: Make that itinerary more budget-friendly

--- Agent Tool Trace ---
  1. research_destination({'destination': 'Dubai'})
  2. get_weather({'city': 'Dubai'})
  3. get_travel_costs({'destination': 'Dubai', 'days': 8})
     → [research_destination] result: WIKIPEDIA OVERVIEW — DUBAI  Dubai is the most populous city in the United Arab Emirates and the capital of the Emirate o...
     → [get_weather] result: CURRENT WEATHER — DUBAI, United Arab Emirates Temperature: 34°C | Humidity: 49% Conditions: Sunny...
     → [get_travel_costs] result: COST ESTIMATE — DUBAI (8 days) [exact city data] Daily: $298/day | Trip total: $2384 USD...
  4. get_travel_costs({'destination': 'Dubai', 'days': 8})
     → [get_travel_costs] result: COST ESTIMATE — DUBAI (8 days) [exact city data] Daily: $298/day | Trip total: $2384 USD...
---

TravelBot (5.5s):
Here's a revised, more budget-friendly itinerary for your 8-day trip to Dubai:

### Trip Highlights
- **Burj Khalifa**: Visit the observation deck for st

"Here's a revised, more budget-friendly itinerary for your 8-day trip to Dubai:\n\n### Trip Highlights\n- **Burj Khalifa**: Visit the observation deck for stunning views (consider going at sunset for a cheaper ticket).\n- **Dubai Mall**: Explore without spending much; enjoy the fountain show and window shopping.\n- **Desert Safari**: Look for group deals or book in advance for discounts.\n- **Public Beaches**: Enjoy the sun at free public beaches like Kite Beach.\n- **Old Dubai**: Visit the Dubai Museum and explore the spice and gold souks for a taste of local culture.\n\n### Suggested Budget-Friendly Itinerary\n**Day 1**: Arrival in Dubai, check into a budget hotel or hostel, and explore your local area.  \n**Day 2**: Visit Burj Khalifa (consider a morning visit for lower rates) and spend the afternoon at Dubai Mall. Enjoy the evening fountain show.  \n**Day 3**: Free day at Kite Beach; pack a picnic to save on food costs.  \n**Day 4**: Explore Old Dubai, including the Dubai Museum (e

## Step 10: Pipeline vs Agentic — Key Takeaways

| Aspect | `mcp_langgraph_travel_planner.ipynb` | This notebook |
|---|---|---|
| Pattern | Deterministic pipeline | Autonomous agent (ReAct) |
| Who picks tools? | Graph edges (fixed) | LLM at runtime |
| Tool calls per query | Exactly 3 (always) | Usually 3, but agent may vary |
| Observability | `node_log` per step | Tool trace from message history |
| Multi-turn | Re-invoke full graph | `thread_id` + `InMemorySaver` |
| Best for | Production workflows, audits | Exploratory queries, flexible dialogue |
| MCP server | Same 3 tools | Same 3 tools |

**When to use agentic:** open-ended questions, follow-ups, or when tool order should adapt to the query.

**When to use pipeline:** regulated flows, predictable cost/latency, or when every step must always run.

## Step 11: Interactive Demo (optional)

In [13]:
print("MCP Travel Agent — type a question or 'quit' to exit\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in {"quit", "exit", "q"}:
        print("Goodbye!")
        break
    await plan_trip(user_input, thread_id="travel-interactive")

MCP Travel Agent — type a question or 'quit' to exit


User: what is ur name

--- Agent Tool Trace ---
  (no tool calls recorded)
---

TravelBot (1.2s):
I am TravelBot, your expert AI travel planner. How can I assist you with your travel plans today?


User: i want to travel from kashmir to kanyakumari in 5 days

--- Agent Tool Trace ---
  1. research_destination({'destination': 'Kanyakumari'})
  2. get_weather({'city': 'Kanyakumari'})
  3. get_travel_costs({'destination': 'Kanyakumari', 'days': 5})
     → [research_destination] result: WIKIPEDIA OVERVIEW — KANYAKUMARI  Kanyakumari is a town and municipality in Kanyakumari district in the state of Tamil N...
     → [get_weather] result: CURRENT WEATHER — KANYAKUMARI, India Temperature: 33°C | Humidity: 41% Conditions: Overcast ...
     → [get_travel_costs] result: COST ESTIMATE — KANYAKUMARI (5 days) [global average] Daily: $193/day | Trip total: $965 USD...
---

TravelBot (6.3s):
Here's a comprehensive travel plan for your trip from K